In [1]:
import pandas as pd
import torch 
import numpy as np
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load the extracted data points
df = pd.read_csv('gestures.csv')

X = df.drop('Labels', axis=1).values
y = df['Labels'].values

num_classes = len(np.unique(y))
print(f'Total data row: {len(df)} with {num_classes} gesture classes')

Total data row: 3064 with 6 gesture classes


In [3]:
# Train, Test and Split (Train: 80%, Test: 20%)
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42 ,stratify=y)

# Standardize the data points
scaler = StandardScaler()

train_X = scaler.fit_transform(train_X)
test_X = scaler.transform(test_X)

# Convert to pytorch tensor
train_X_t = torch.FloatTensor(train_X)
test_X_t = torch.FloatTensor(test_X)
train_y_t = torch.LongTensor(train_y)
test_y_t = torch.LongTensor(test_y)

In [4]:
# Define LightWeight 3-layers ANN 
class GestureANN(nn.Module):
    def __init__(self, input_dim=42, num_classes=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)


model = GestureANN(input_dim=42, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# Train Model
epochs = 60
print('-----Training Model------')

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(train_X_t)
    loss = criterion(outputs, train_y_t)
    loss.backward(),
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.4f}')

-----Training Model------
Epoch [10/60] - Loss: 1.6039
Epoch [20/60] - Loss: 1.3562
Epoch [30/60] - Loss: 1.0713
Epoch [40/60] - Loss: 0.7925
Epoch [50/60] - Loss: 0.5346
Epoch [60/60] - Loss: 0.3305


In [6]:
# Evaluate Accuracy
model.eval()
with torch.no_grad():
    test_outputs = model(test_X_t)
    _, predicted = torch.max(test_outputs, 1)
    correct = (predicted == test_y_t).sum().item()
    accuracy = (correct / len(test_y_t)) * 100
    print(f"\n✅ Final Test Accuracy: {accuracy:.2f}%")

# 6. Save Model Checkpoint
torch.save({'model_state': model.state_dict(), 'scaler': scaler, 'num_classes': num_classes}, "gesture_ann_model.pth")
print("💾 Model saved successfully as 'gesture_ann_model.pth'!")


✅ Final Test Accuracy: 98.53%
💾 Model saved successfully as 'gesture_ann_model.pth'!
